In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# 03. PEMC Demo\n",
    "Prediction-Enhanced Monte Carlo for an Asian option.\n",
    "\n",
    "Both samplers return `(payoffs, features)`. The cheap sampler discards the payoff, so the independent samples only need the predictor. In this numpy demo both samplers cost about the same, so the gain you will see is in **variance**, not wall-clock time."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import sys\n",
    "import os\n",
    "sys.path.insert(0, os.path.abspath('..'))\n",
    "\n",
    "import numpy as np\n",
    "import torch\n",
    "import matplotlib.pyplot as plt\n",
    "\n",
    "from src.pipeline.pemc.estimator import PEMCConfig, NeuralPredictor, PEMCEstimator\n",
    "from src.pipeline.pemc.trainer import PEMCTrainer\n",
    "from src.pipeline.pemc.option_models import create_sampler\n",
    "from src.pipeline.pemc.variance_analyzer import VarianceAnalyzer\n",
    "\n",
    "%matplotlib inline\n",
    "np.random.seed(0)\n",
    "torch.manual_seed(0)\n",
    "device = 'cuda' if torch.cuda.is_available() else 'cpu'\n",
    "print('[INFO] Using device:', device)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "sampler = create_sampler('asian')   # sampler(n) -> (payoffs, features[n, 7])\n",
    "\n",
    "def expensive_sampler(n):\n",
    "    return sampler(n)\n",
    "\n",
    "def cheap_sampler(n):\n",
    "    # Same feature distribution as the coupled samples (required for an unbiased estimate).\n",
    "    # No re-seeding here: re-seeding made every chunk identical in the old notebook.\n",
    "    _, feats = sampler(n)\n",
    "    return None, feats"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "config = PEMCConfig(n_coupled=1000, n_independent=50000)\n",
    "predictor = NeuralPredictor(input_dim=7)\n",
    "trainer = PEMCTrainer(predictor, config, device=device)\n",
    "\n",
    "features, payoffs = trainer.generate_training_data(sampler, n_samples=20000)\n",
    "print('[INFO] Features shape:', features.shape)\n",
    "print('[INFO] Payoffs shape:', payoffs.shape)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "history = trainer.train(features=features, payoffs=payoffs, epochs=50, batch_size=64)\n",
    "\n",
    "plt.figure(figsize=(10, 5))\n",
    "plt.plot(history['train_loss'], label='Train')\n",
    "plt.plot(history['val_loss'], label='Validation')\n",
    "plt.xlabel('Epoch')\n",
    "plt.ylabel('MSE')\n",
    "plt.yscale('log')\n",
    "plt.title('Neural predictor training')\n",
    "plt.legend()\n",
    "plt.tight_layout()\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "estimator = PEMCEstimator(trainer.predictor, config)\n",
    "estimate, variance, metrics = estimator.estimate(expensive_sampler, cheap_sampler)\n",
    "\n",
    "print('PEMC estimate:', round(estimate, 4))\n",
    "print('Variance of estimate:', variance)\n",
    "print('Variance reduction vs plain MC (same 1000 expensive samples):', round(metrics['variance_reduction'] * 100, 2), '%')\n",
    "print('95% CI:', round(metrics['ci_lower'], 4), 'to', round(metrics['ci_upper'], 4))"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Reference value from a large plain MC run (400k paths, in chunks to keep memory low)\n",
    "total, n_total = 0.0, 0\n",
    "for _ in range(20):\n",
    "    p, _ = sampler(20000)\n",
    "    total += float(p.sum())\n",
    "    n_total += len(p)\n",
    "reference = total / n_total\n",
    "print('Reference price (400k paths):', round(reference, 4))\n",
    "print('PEMC estimate               :', round(estimate, 4))"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Empirical variance: repeat both estimators with the SAME number of expensive samples (1000)\n",
    "R = 30\n",
    "small = PEMCConfig(n_coupled=1000, n_independent=20000)\n",
    "est_small = PEMCEstimator(trainer.predictor, small)\n",
    "\n",
    "pemc_estimates, mc_estimates = [], []\n",
    "for _ in range(R):\n",
    "    e, _, _ = est_small.estimate(expensive_sampler, cheap_sampler)\n",
    "    pemc_estimates.append(e)\n",
    "    p, _ = sampler(1000)\n",
    "    mc_estimates.append(float(p.mean()))\n",
    "\n",
    "analyzer = VarianceAnalyzer()\n",
    "vm = analyzer.analyze_variance_reduction(np.array(mc_estimates), np.array(pemc_estimates))\n",
    "print('Plain MC std :', round(float(np.std(mc_estimates, ddof=1)), 5))\n",
    "print('PEMC std     :', round(float(np.std(pemc_estimates, ddof=1)), 5))\n",
    "print('Empirical variance reduction:', round(vm.variance_reduction * 100, 1), '% (R = 30 runs, so this is noisy)')\n",
    "\n",
    "fig = analyzer.plot_variance_comparison(np.array(mc_estimates), np.array(pemc_estimates))\n",
    "fig"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "os.makedirs('../data/models', exist_ok=True)\n",
    "trainer.save_predictor('../data/models/pemc_predictor.pth')\n",
    "print('[INFO] Predictor saved')"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {"display_name": "Python 3", "language": "python", "name": "python3"},
  "language_info": {"name": "python"}
 },
 "nbformat": 4,
 "nbformat_minor": 4
}